# Queue-Server-Dispatcher-Attacker

In [1]:
from pringles.simulator import Simulator
mySimulator = Simulator(cdpp_bin_path='bin/', user_models_dir='src/')

In [2]:
atomics = dict([(atomic.__name__, atomic) for atomic in mySimulator.atomic_registry.discovered_atomics])
Server = atomics['Server']
Dispatcher = atomics['Dispatcher']
Attacker = atomics['Attacker']
ServerQueue = atomics['ServerQueue']

In [3]:
number_of_servers = 5


dispatcherDefaultConfig = {
    'numberOfServers': number_of_servers
}
for i in range(number_of_servers):
    dispatcherDefaultConfig['server' + str(i)] = 'free'


serverDefaultConfig = {
    'distribution': 'exponential',
    'mean': 0.001,
    'setupTime': '00:00:10:00'
}

serverOnDefaultConifg = {
    'initialStatus': 'free',
    **serverDefaultConfig
}

serverOffDefaultConifg = {
    'initialStatus': 'off',
    **serverDefaultConfig
}

serverQueueDefaultConfig = {
    'size': 1000,
    'currentSizeFrequency': '00:00:00:10'
}

attackerDefaultConfig = {
    'file': 'attack-data/diffsShort.txt'
}

In [4]:
from pringles.models.errors import PortNotFoundException

dispatcher = Dispatcher('dispatcher', **dispatcherDefaultConfig)
queue = ServerQueue('queue', **serverQueueDefaultConfig)
servers = {}
for i in range(number_of_servers):
    server_name = 'server' + str(i)
    servers[i] = Server(server_name, **serverOnDefaultConifg)
    try:
        dispatcher.get_port(server_name)
    except PortNotFoundException:
        dispatcher.add_outport(server_name)

attacker = Attacker('attacker', **attackerDefaultConfig)

In [5]:
print("--Dispatcher ports")
print("Inport names: ", [port.name for port in dispatcher.inports])
print("Outport names: ", [port.name for port in dispatcher.outports])

print("--Server ports--")
print("Inport names: ", [port.name for port in servers[0].inports])
print("Outport names: ", [port.name for port in servers[0].outports])

print("--ServerQueue ports--")
print("Inport names: ", [port.name for port in queue.inports])
print("Outport names: ", [port.name for port in queue.outports])

print("--Attacker ports--")
print("Inport names: ", [port.name for port in attacker.inports])
print("Outport names: ", [port.name for port in attacker.outports])

--Dispatcher ports
Inport names:  ['newJob', 'jobDone', 'serverStatus']
Outport names:  ['requestJob', 'server0', 'server1', 'server2', 'server3', 'server4']
--Server ports--
Inport names:  ['job', 'powerSignal']
Outport names:  ['done', 'ready']
--ServerQueue ports--
Inport names:  ['in', 'emit']
Outport names:  ['out', 'discarded', 'queueLoad', 'loadAvg']
--Attacker ports--
Inport names:  []
Outport names:  ['attack']


In [6]:
from pringles.models import Coupled

subcomponents = [dispatcher, queue, attacker] + list(servers.values())
    
top_model = Coupled(name='top', subcomponents=subcomponents)

# adding top inports
top_model.add_inport("serverStatus")
top_model.add_inport("queueLoad")
top_model.add_outport('discarded')
top_model.add_outport('jobDone')

# # adding serverStatus coupling to inport of dispatcher
top_model.add_coupling('serverStatus', dispatcher.get_port("serverStatus"))

# adding couplings between queue and dispatcher
top_model.add_coupling(queue.get_port("out"), dispatcher.get_port("newJob"))
top_model.add_coupling(dispatcher.get_port("requestJob"), queue.get_port("emit"))

# adding coupling between attacker and queue
top_model.add_coupling(attacker.get_port('attack'), queue.get_port('in'))
                                                                   
# adding couplings between dispatcher and servers
for i in range(number_of_servers):
    server_name = 'server' + str(i)
    top_model.add_coupling(dispatcher.get_port(server_name), servers[i].get_port('job'))
    top_model.add_coupling(servers[i].get_port('done'), dispatcher.get_port('jobDone'))
    top_model.add_coupling(servers[i].get_port('done'), 'jobDone')

# adding coupling between queue discarded port and top
top_model.add_coupling(queue.get_port('discarded'), 'discarded')

top_model

In [7]:
from pringles.simulator import Simulation, Event
from pringles.utils import VirtualTime
import os

working_dir = 'sim_results/queue-server-dispatcher-attacker'

# Create the directory structure if it does not exist
if not os.path.exists(working_dir):
    os.makedirs(working_dir)


a_simulation = Simulation(top_model = top_model, 
                          duration = VirtualTime.of_minutes(10),
                          working_dir = working_dir
                         )

results = mySimulator.run_simulation(a_simulation)

In [8]:
print(results.get_process_output())

PCD++: A Tool to Implement n-Dimensional Cell-DEVS models
Version 3.0 - March 2003
Troccoli A., Rodriguez D., Wainer G., Barylko A., Beyoglonian J., Lopez A.
-----------------------------------------------------------------------------
PCD++ Extended States: An extended and improved version of CD++ for Cell-DEVS
Version 4.1.2 - December 2018
Santi L., Castro, R., Pimás, J.
-----------------------------------------------------------------------------
Discrete Event Simulation Lab
Departamento de Computación
Facultad de Ciencias Exactas y Naturales
Universidad de Buenos Aires, Argentina
-----------------------------------------------------------------------------
Compiled for standalone simulation


Loading models from sim_results/queue-server-dispatcher-attacker/2024-03-29-210742-7555f4e1040b49088b81e585b4ce1c52/top_model
Loading events from 
Running parallel simulation. Reading models partition from 
Model partition details output to: /dev/null*
Message log: sim_results/queue-server-di

In [10]:
display(results.output_df.head(100))

,time,port,value
0,00:00:00:000,jobdone,0.0
1,00:00:00:000,jobdone,2.0
2,00:00:00:000,jobdone,5.0
3,00:00:00:001,jobdone,3.0
4,00:00:00:001,jobdone,1.0
...,...,...,...
95,00:00:00:021,jobdone,97.0
96,00:00:00:022,jobdone,99.0
97,00:00:00:022,jobdone,93.0
98,00:00:00:022,jobdone,101.0


In [11]:
print(results.logs_dfs.keys(),'\n\n')
display(results.logs_dfs['ParallelRoot'].head())

dict_keys(['server4', 'server3', 'server2', 'server1', 'server0', 'attacker', 'queue', 'dispatcher', 'top', 'ParallelRoot']) 




,0,1,message_type,time,model_origin,port,value,model_dest
0,0,L,Y,00:00:00:000,top(09),jobdone,0.0,ParallelRoot(00)
1,0,L,Y,00:00:00:000,top(09),jobdone,2.0,ParallelRoot(00)
2,0,L,Y,00:00:00:000,top(09),jobdone,5.0,ParallelRoot(00)
3,0,L,Y,00:00:00:001,top(09),jobdone,3.0,ParallelRoot(00)
4,0,L,Y,00:00:00:001,top(09),jobdone,1.0,ParallelRoot(00)


In [12]:
# Veamos que le llegan al server 0 los primneros 2 jobs, hasta que lo apagan en segundo 40
display(results.logs_dfs['server0'])

,0,1,message_type,time,model_origin,port,value,model_dest
0,0,L,X,00:00:00:000,top(09),job,0.0,server0(04)
1,0,L,X,00:00:00:000,top(09),job,5.0,server0(04)
2,0,L,X,00:00:00:000,top(09),job,7.0,server0(04)
3,0,L,X,00:00:00:001,top(09),job,11.0,server0(04)
4,0,L,X,00:00:00:001,top(09),job,12.0,server0(04)
...,...,...,...,...,...,...,...,...
2208,0,L,X,00:00:04:798,top(09),job,9649.0,server0(04)
2209,0,L,X,00:00:04:799,top(09),job,9657.0,server0(04)
2210,0,L,X,00:00:04:811,top(09),job,9660.0,server0(04)
2211,0,L,X,00:00:04:812,top(09),job,9665.0,server0(04)


In [13]:
# Veamos que el siguiente job le llega al server 1 en el segundo 42, y luego lo apagan en el segundo 43
# Y se espera a que termine para apagarlo
display(results.logs_dfs['server1'])

,0,1,message_type,time,model_origin,port,value,model_dest
0,0,L,X,00:00:00:000,top(09),job,1.0,server1(05)
1,0,L,X,00:00:00:001,top(09),job,9.0,server1(05)
2,0,L,X,00:00:00:003,top(09),job,17.0,server1(05)
3,0,L,X,00:00:00:003,top(09),job,19.0,server1(05)
4,0,L,X,00:00:00:003,top(09),job,21.0,server1(05)
...,...,...,...,...,...,...,...,...
2084,0,L,X,00:00:04:798,top(09),job,9648.0,server1(05)
2085,0,L,X,00:00:04:798,top(09),job,9653.0,server1(05)
2086,0,L,X,00:00:04:811,top(09),job,9661.0,server1(05)
2087,0,L,X,00:00:04:814,top(09),job,9674.0,server1(05)


In [14]:
# Por último cuando el 0 y el 1 están apagados, cuando llega el último job en el segundo 44, se lo envía al server 2
# que lo termina (ver logs dispatcher)
display(results.logs_dfs['server2'])

,0,1,message_type,time,model_origin,port,value,model_dest
0,0,L,X,00:00:00:000,top(09),job,2.0,server2(06)
1,0,L,X,00:00:00:000,top(09),job,6.0,server2(06)
2,0,L,X,00:00:00:001,top(09),job,10.0,server2(06)
3,0,L,X,00:00:00:003,top(09),job,18.0,server2(06)
4,0,L,X,00:00:00:004,top(09),job,24.0,server2(06)
...,...,...,...,...,...,...,...,...
1984,0,L,X,00:00:04:799,top(09),job,9656.0,server2(06)
1985,0,L,X,00:00:04:799,top(09),job,9659.0,server2(06)
1986,0,L,X,00:00:04:811,top(09),job,9662.0,server2(06)
1987,0,L,X,00:00:04:813,top(09),job,9670.0,server2(06)


In [15]:
display(results.logs_dfs['dispatcher'])

,0,1,message_type,time,model_origin,port,value,model_dest
0,0,L,X,00:00:00:000,top(09),newjob,1.0,dispatcher(01)
1,0,L,X,00:00:00:000,top(09),newjob,1.0,dispatcher(01)
2,0,L,X,00:00:00:000,top(09),newjob,1.0,dispatcher(01)
3,0,L,X,00:00:00:000,top(09),newjob,1.0,dispatcher(01)
4,0,L,X,00:00:00:000,top(09),newjob,1.0,dispatcher(01)
...,...,...,...,...,...,...,...,...
19349,0,L,X,00:00:04:816,top(09),newjob,1.0,dispatcher(01)
19350,0,L,X,00:00:04:816,top(09),newjob,1.0,dispatcher(01)
19351,0,L,X,00:00:04:817,top(09),jobdone,9675.0,dispatcher(01)
19352,0,L,X,00:00:04:817,top(09),jobdone,9669.0,dispatcher(01)
